In [1]:
import os

In [2]:
%pwd

'C:\\Users\\nisch\\Downloads\\Chicken-Disease-Classification-Projects-main\\Chicken-Disease-Classification-Projects-main\\research'

In [3]:
from pathlib import Path
import os

_cwd = Path.cwd().resolve()
if _cwd.name == "research":
    os.chdir(_cwd.parent)

In [4]:
%pwd

'C:\\Users\\nisch\\Downloads\\Chicken-Disease-Classification-Projects-main\\Chicken-Disease-Classification-Projects-main'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list



@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [6]:
import sys
import os
import unittest

if not hasattr(unittest.TestCase, "assertRaisesRegexp"):
    unittest.TestCase.assertRaisesRegexp = unittest.TestCase.assertRaisesRegex

sys.path.insert(0, os.path.abspath("src"))

from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

os.environ.setdefault("KERAS_BACKEND", "torch")
try:
    import tensorflow as tf
except ModuleNotFoundError:
    import keras as _keras

    class _TF:
        keras = _keras

    tf = _TF()

In [7]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])


    
    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks
        model_ckpt_dir = os.path.dirname(config.checkpoint_model_filepath)
        create_directories([
            Path(model_ckpt_dir),
            Path(config.tensorboard_root_log_dir)
        ])

        prepare_callback_config = PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir=Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath)
        )

        return prepare_callback_config
    




    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chicken-fecal-images")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [8]:
import time

In [9]:
class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config


    
    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}",
        )
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)
    

    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=str(self.config.checkpoint_model_filepath),
            save_best_only=True,
            monitor="val_loss",
            mode="min",
        )


    def get_tb_ckpt_callbacks(self):
        import importlib.util

        callbacks = []
        if importlib.util.find_spec("tensorflow") is not None:
            callbacks.append(self._create_tb_callbacks)
        callbacks.append(self._create_ckpt_callbacks)
        return callbacks


In [10]:
# TensorFlow/Keras: cell 5; time: cell 7
pass

In [11]:
from keras_preprocessing.image import ImageDataGenerator


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config
    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            str(self.config.updated_base_model_path)
        )
        # Fresh compile for Keras 3 + PyTorch after load (optimizer state is not portable)
        self.model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"],
        )
    
    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=str(self.config.training_data),
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=str(self.config.training_data),
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(str(path))


    @staticmethod
    def _iter_batches(flow):
        while True:
            yield flow.next()

    def train(self, callback_list: list):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self._iter_batches(self.train_generator),
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self._iter_batches(self.valid_generator),
            callbacks=callback_list
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )



In [12]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(
        callback_list=callback_list
    )
    
except Exception as e:
    raise e

[2026-03-28 23:12:23,352: INFO: common: yaml file: config\config.yaml loaded successfully]


[2026-03-28 23:12:23,355: INFO: common: yaml file: params.yaml loaded successfully]


[2026-03-28 23:12:23,356: INFO: common: created directory at: artifacts]


[2026-03-28 23:12:23,358: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir]


[2026-03-28 23:12:23,359: INFO: common: created directory at: artifacts\prepare_callbacks\tensorboard_log_dir]


...

18/19 ━━━━━━━━━━━━━━━━━━━━ 7s 8s/step - accuracy: 0.4129 - loss: 7.2632 

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.4166 - loss: 7.2876

[2026-03-28 23:15:06,740: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]


19/19 ━━━━━━━━━━━━━━━━━━━━ 163s 9s/step - accuracy: 0.4836 - loss: 7.7262 - val_accuracy: 0.6094 - val_loss: 6.2961


[2026-03-28 23:15:07,838: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
